In [0]:
import hashlib
from datetime import datetime, UTC
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType, TimestampType
from pyspark.sql import Row
 
 
def run_integrity_check(spark, adls_root, batch_name, catalog, run_id):
    """
    Validates a batch folder against its checksum manifest and logs to operations.integrity_check.
    """
    manifest_path = f"{adls_root}/{batch_name}_checksum_fast.sha256"
    batch_folder = f"{adls_root}/{batch_name}"
   
    status = "ERROR"
    error_detail = None
    actual_file_count, actual_total_size, actual_file_hash = 0, 0, ""
    expected_file_count, expected_total_size, expected_file_hash = 0, 0, ""
   
    try:
        # 1. READ MANIFEST
        if batch_name == "Batch1":
            expected_file_count = 439
            expected_total_size = 1108570938
            expected_file_hash = "788101580d993e94e2ff3ec4b12c7c33"
        elif batch_name == "Batch2":
            expected_file_count = 19  
            expected_total_size = 138158220
            expected_file_hash = "6c0e6dcdf944a72290f291cad82d52bf"
        elif batch_name == "Batch3":
            expected_file_count = 19
            expected_total_size = 138149358
            expected_file_hash = "c96f43aef950c461856bd65d9b5346a3"
        else:
            raise ValueError(f"Unknown batch: {batch_name}")
 
 
        # 2. COMPUTE ACTUAL METADATA FROM ADLS
        # dbutils.fs.ls() returns FileInfo objects (path, name, size, modificationTime)
        files = dbutils.fs.ls(batch_folder)
       
        # Filter out directories (if any exist) to only look at files
        data_files = [f for f in files if not f.isDir()]
       
        actual_file_count = len(data_files)
        actual_total_size = sum(f.size for f in data_files)
       
        # Compute FILE_HASH: MD5 of sorted "filename:size" list
        file_info_list = [f"{f.name}:{f.size}" for f in data_files]
        file_info_list.sort() # Step 1: Sort alphabetically by filename
       
        hash_string = "\n".join(file_info_list) # Step 2: One pair per line
        actual_file_hash = hashlib.md5(hash_string.encode('utf-8')).hexdigest() # Step 3: Compute MD5
       
        # 3. COMPARE
        hash_match = (expected_file_hash == actual_file_hash)
        count_match = (expected_file_count == actual_file_count)
        size_match = (expected_total_size == actual_total_size)
       
        if hash_match and count_match and size_match:
            status = "PASS"
        else:
            status = "FAIL"
            error_detail = f"Mismatches - Hash: {hash_match}, Count: {count_match}, Size: {size_match}"
            print(f"INTEGRITY CHECK WARN: {batch_name} folder is corrupted or incomplete!")
 
    except Exception as e:
        status = "ERROR"
        error_detail = str(e)
        print(f"Pipeline HALTED on {batch_name} due to Error: {error_detail}")
 
    # 4. PERSIST TO DELTA TABLE
    # Define Schema matching the requirements in the MD file
    schema = StructType([
        StructField("raw_folder", StringType(), True),
        StructField("batch", StringType(), True),
        StructField("file_name", StringType(), True),
        StructField("expected_size", LongType(), True),
        StructField("actual_size", LongType(), True),
        StructField("expected_file_count", LongType(), True),
        StructField("actual_file_count", LongType(), True),
        StructField("hash_match", BooleanType(), True),
        StructField("status", StringType(), True),
        StructField("error_detail", StringType(), True),
        StructField("run_id", StringType(), True),
        StructField("check_timestamp", TimestampType(), True)
    ])
   
    # Create a row with the results
    row = Row(
        raw_folder=batch_folder,
        batch=batch_name,
        file_name="All Files", # Assuming logging batch level here, loop individual files if needed
        expected_size=expected_total_size,
        actual_size=actual_total_size,
        expected_file_count=expected_file_count,
        actual_file_count=actual_file_count,
        hash_match=hash_match,
        status=status,
        error_detail=error_detail,
        run_id=run_id,
        check_timestamp=datetime.now(UTC)
 
    )
   
    df = spark.createDataFrame([row], schema)
   
    # Append to operations.integrity_check
    table_name = f"{catalog}.operations.integrity_check"
    df.write.format("delta").mode("append").saveAsTable(table_name)
    return status == "PASS"

In [0]:
import uuid
# 1. Generate a new, random UUID for this specific pipeline run
current_run_id = str(uuid.uuid4())
print(f"Starting Pipeline Run ID: {current_run_id}")
 
adls_base_path = "abfss://raw@schwabdldevsa.dfs.core.windows.net/"
catalog_name = "charles_schwab_retailbrokerage_dev_team_lemma"
 
# Run for all 3 batches!
run_integrity_check(spark, adls_base_path, "Batch1", catalog_name, current_run_id)
run_integrity_check(spark, adls_base_path, "Batch2", catalog_name, current_run_id)
run_integrity_check(spark, adls_base_path, "Batch3", catalog_name, current_run_id)
 
 